In [74]:
import re
import json
from pathlib import Path
from collections import defaultdict

In [75]:
CODE_DIR = 'dataset/storybook/src/stories'
CODE_DIR_PATH = Path(CODE_DIR)

METHODS = ['gt', 'a', 'b1', 'b2', 'b3', 'c1', 'c2', 'c3']
EVAL_METHODS = ['a', 'b1', 'b2', 'b3', 'c1', 'c2', 'c3']

COMPLEXITIES = ['simple', 'medium', 'hard']

PRIMEVUE_COMPONENTS = {
    'Accordion', 'AccordionPanel', 'AccordionHeader', 'AccordionContent',
    'Avatar', 'AvatarGroup',
    'Badge', 'Breadcrumb', 'Button',
    'Card', 'Checkbox', 'Column', 'ColumnGroup',
    'DataTable', 'DatePicker', 'Dialog', 'Divider',
    'InputNumber', 'InputText',
    'Menu',
    'OverlayBadge',
    'Password', 'Popover', 'ProgressBar',
    'RadioButton', 'Row',
    'Select', 'Skeleton', 'Slider',
    'Tab', 'TabList', 'TabPanel', 'TabPanels', 'Tabs', 'Tag', 'Textarea',
    'ToggleSwitch',
}

SKIP_PROPS = {
    'class', 'style', 'id', 'ref', 'key',
    'v-model', 'v-if', 'v-else', 'v-for', 'v-show',
    'v-model:visible',   # Dialog
}


## Load reference and generated code files in groups by method and complexity

In [76]:
CODE_FILES: dict[str, str] = {}

for code_file in CODE_DIR_PATH.rglob('*.vue'):
    key = f'{code_file.parent.name}/{code_file.stem}'.lower()

    CODE_FILES[key] = code_file.read_text(encoding='utf-8', errors='ignore')

print(f'Load code files: {len(CODE_FILES)}')

GROUPED_FILES: dict[str, dict[str, dict[str, str]]] = {}

FILE_PATTERN = re.compile(r'^(\d+)(?:-([a-z0-9]+))?$')

for key, content in CODE_FILES.items():
    complexity, stem = key.split('/', 1)
    match = FILE_PATTERN.match(stem)

    if not match:
        continue

    index, method = match.groups()

    if method is None:
        method = 'gt'

    index = index.zfill(2)

    if method not in METHODS:
        continue

    GROUPED_FILES.setdefault(method, {})
    GROUPED_FILES[method].setdefault(complexity, {})
    GROUPED_FILES[method][complexity][index] = content

print('Grouped files:')
for method in METHODS:
    counts = {c: len(v) for c, v in GROUPED_FILES[method].items()}
    total  = sum(counts.values())

    print(f'  {method:4s}  {total:3d} files  {counts}')

Load code files: 240
Grouped files:
  gt     30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  a      30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  b1     30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  b2     30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  b3     30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  c1     30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  c2     30 files  {'hard': 10, 'medium': 10, 'simple': 10}
  c3     30 files  {'hard': 10, 'medium': 10, 'simple': 10}


## Component F1-Scores (P,R,F1)

In [77]:
def extract_components(sfc: str) -> set[str]:
    """Extracts all PrimeVue component names from a Vue SFC.

    Two sources, combined:
    1. Import statements:  import Button from 'primevue/button'
    2. Template tags:  <Button ...>, <DataTable ...>
    """
    components = set()

    # 1. From import rows
    for m in re.finditer(r'import\s+(\w+)\s+from\s+[\'"]primevue/', sfc):
        components.add(m.group(1))

    # 2. From template tags (opening tags with an uppercase letter)
    template_match = re.search(
        r'<template>(.*?)</template>', sfc, re.DOTALL
    )

    if template_match:
        template = template_match.group(1)

        for m in re.finditer(r'<([A-Z][a-zA-Z]+)', template):
            tag = m.group(1)

            if tag in PRIMEVUE_COMPONENTS:
                components.add(tag)

    return components

In [78]:
def compute_f1(generated: set[str], ground_truth: set[str]) -> dict:
    """Calculates precision, recall, and F1 score between two sets of components"""
    if not generated and not ground_truth:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0,
                'true_positives': set(), 'false_positives': set(),
                'false_negatives': set()}

    tp = generated & ground_truth        # correctly identified
    fp = generated - ground_truth        # hallucinating / incorrect
    fn = ground_truth - generated        # overlooked / missing

    precision = len(tp) / len(generated)     if generated     else 0.0
    recall    = len(tp) / len(ground_truth)  if ground_truth  else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    return {
        'precision':       round(precision, 4),
        'recall':          round(recall, 4),
        'f1':              round(f1, 4),
        'true_positives':  tp,
        'false_positives': fp,
        'false_negatives': fn,
    }

In [79]:
f1_results: list[dict] = []

for complexity in COMPLEXITIES:
    gt_files = GROUPED_FILES.get('gt', {}).get(complexity, {})

    for index, gt_sfc in gt_files.items():
        gt_comps = extract_components(gt_sfc)

        for method in EVAL_METHODS:
            gen_sfc = GROUPED_FILES.get(method, {}).get(complexity, {}).get(index)

            if gen_sfc is None:
                print(f'MISSING {complexity}/{index} [{method}]')

                continue

            gen_comps = extract_components(gen_sfc)
            scores = compute_f1(gen_comps, gt_comps)

            print(f'{complexity}/{index} [{method}]  P={scores["precision"]:.2f}  R={scores["recall"]:.2f}  F1={scores["f1"]:.2f}')

            f1_results.append({
                'mockup':      f'{complexity}-{index}',
                'complexity':   complexity,
                'index':        index,
                'method':       method,
                'precision':    scores['precision'],
                'recall':       scores['recall'],
                'f1':           scores['f1'],
                'tp':           sorted(scores['true_positives']),
                'fp':           sorted(scores['false_positives']),
                'fn':           sorted(scores['false_negatives']),
                'gt_count':     len(gt_comps),
                'gen_count':    len(gen_comps),
            })

print(f'Computed: {len(f1_results)} F1-Scores')

simple/01 [a]  P=1.00  R=1.00  F1=1.00
simple/01 [b1]  P=1.00  R=1.00  F1=1.00
simple/01 [b2]  P=1.00  R=1.00  F1=1.00
simple/01 [b3]  P=1.00  R=1.00  F1=1.00
simple/01 [c1]  P=0.25  R=0.50  F1=0.33
simple/01 [c2]  P=1.00  R=1.00  F1=1.00
simple/01 [c3]  P=1.00  R=1.00  F1=1.00
simple/10 [a]  P=1.00  R=0.75  F1=0.86
simple/10 [b1]  P=1.00  R=1.00  F1=1.00
simple/10 [b2]  P=1.00  R=0.75  F1=0.86
simple/10 [b3]  P=1.00  R=0.75  F1=0.86
simple/10 [c1]  P=0.86  R=0.75  F1=0.80
simple/10 [c2]  P=1.00  R=0.75  F1=0.86
simple/10 [c3]  P=1.00  R=0.88  F1=0.93
simple/02 [a]  P=1.00  R=1.00  F1=1.00
simple/02 [b1]  P=0.50  R=0.50  F1=0.50
simple/02 [b2]  P=1.00  R=1.00  F1=1.00
simple/02 [b3]  P=1.00  R=1.00  F1=1.00
simple/02 [c1]  P=0.50  R=0.50  F1=0.50
simple/02 [c2]  P=1.00  R=1.00  F1=1.00
simple/02 [c3]  P=1.00  R=1.00  F1=1.00
simple/03 [a]  P=1.00  R=1.00  F1=1.00
simple/03 [b1]  P=1.00  R=1.00  F1=1.00
simple/03 [b2]  P=1.00  R=1.00  F1=1.00
simple/03 [b3]  P=1.00  R=1.00  F1=1.00
simp

In [80]:
# --- Level 1: Average per method ---
print(f'\n{"Method":6s} {"Ø Precision":>12s} {"Ø Recall":>10s} {"Ø F1":>8s} {"n":>4s}')
print('-' * 45)

by_method = defaultdict(list)
for r in f1_results:
    by_method[r['method']].append(r)

for method in EVAL_METHODS:
    items = by_method[method]

    if not items:
        continue

    avg_p  = sum(i['precision'] for i in items) / len(items)
    avg_r  = sum(i['recall']    for i in items) / len(items)
    avg_f1 = sum(i['f1']        for i in items) / len(items)

    print(f'{method:6s} {avg_p:12.3f} {avg_r:10.3f} {avg_f1:8.3f} {len(items):4d}')

# --- Level 2: Average per method × complexity (for UF5) ---
print(f'\n{"Method":6s} {"Complexity":12s} {"Ø F1":>8s} {"n":>4s}')
print('-' * 35)

by_method_complexity = defaultdict(list)
for r in f1_results:
    by_method_complexity[(r['method'], r['complexity'])].append(r)

for method in EVAL_METHODS:
    for complexity in COMPLEXITIES:
        items = by_method_complexity[(method, complexity)]

        if not items:
            continue

        avg_f1 = sum(i['f1'] for i in items) / len(items)

        print(f'{method:6s} {complexity:12s} {avg_f1:8.3f} {len(items):4d}')


Method  Ø Precision   Ø Recall     Ø F1    n
---------------------------------------------
a             0.950      0.870    0.897   30
b1            0.930      0.831    0.870   30
b2            0.950      0.891    0.914   30
b3            0.943      0.885    0.907   30
c1            0.747      0.739    0.728   30
c2            0.930      0.888    0.901   30
c3            0.917      0.890    0.895   30

Method Complexity       Ø F1    n
-----------------------------------
a      simple          0.952   10
a      medium          0.922   10
a      hard            0.816   10
b1     simple          0.950   10
b1     medium          0.894   10
b1     hard            0.767   10
b2     simple          0.952   10
b2     medium          0.981   10
b2     hard            0.807   10
b3     simple          0.952   10
b3     medium          0.971   10
b3     hard            0.796   10
c1     simple          0.697   10
c1     medium          0.841   10
c1     hard            0.647   10
c2     simpl

In [81]:
degradation = {}
for method in EVAL_METHODS:
    simple_f1 = sum(i['f1'] for i in by_method_complexity[(method, 'simple')]) \
                / len(by_method_complexity[(method, 'simple')]) \
                if by_method_complexity[(method, 'simple')] else None

    hard_f1   = sum(i['f1'] for i in by_method_complexity[(method, 'hard')]) \
                / len(by_method_complexity[(method, 'hard')]) \
                if by_method_complexity[(method, 'hard')] else None

    degradation[method] = {
        'f1_simple':           round(simple_f1, 4) if simple_f1 else None,
        'f1_hard':             round(hard_f1,   4) if hard_f1   else None,
        'degradation_factor':  round(hard_f1 / simple_f1, 4)
                               if simple_f1 and hard_f1 else None,
    }

report = {
    'metric':      'component_f1',
    'per_method':  {
        method: {
            'avg_precision': round(sum(i['precision'] for i in items) / len(items), 4),
            'avg_recall':    round(sum(i['recall']    for i in items) / len(items), 4),
            'avg_f1':        round(sum(i['f1']        for i in items) / len(items), 4),
            'n':             len(items),
        }
        for method, items in by_method.items()
    },
    'per_method_complexity': {
        f'{m}-{c}': {
            'avg_f1': round(sum(i['f1'] for i in items) / len(items), 4),
            'n':      len(items),
        }
        for (m, c), items in by_method_complexity.items()
    },
    'degradation_factors': degradation,
    'files': f1_results,
}

Path('reports/eval_f1.json').write_text(
    json.dumps(report, indent=4, ensure_ascii=False), encoding='utf-8'
)
print('Saved: reports/eval_f1.json')

Saved: reports/eval_f1.json


## Component-Prop Accuracy

In [82]:
def parse_props(attrs_str: str) -> dict[str, str]:
    """Extracts semantic props from the attribute string of a Vue tag"""
    props = {}

    if not attrs_str:
        return props

    # 1. Dynamic Props: :prop="value"
    for m in re.finditer(r':([a-zA-Z][\w-]*)="([^"]*)"', attrs_str):
        key, val = m.group(1), m.group(2)

        if key not in SKIP_PROPS and not key.startswith('v-'):
            props[f':{key}'] = val

    # 2. Static Props: prop="value"
    for m in re.finditer(r'(?<!:)\b([a-zA-Z][\w-]*)="([^"]*)"', attrs_str):
        key, val = m.group(1), m.group(2)

        if key not in SKIP_PROPS and not key.startswith(('v-', '@')):
            props[key] = val

    # 3. Boolean Props without Value: binary, showButtons, toggleMask
    all_key_positions = {
        m.start() for m in re.finditer(r'[a-zA-Z][\w-]*=', attrs_str)
    }
    for m in re.finditer(r'\b([a-zA-Z][\w-]*)\b', attrs_str):
        if m.start() not in all_key_positions:
            key = m.group(1)

            if key not in SKIP_PROPS and not key.startswith('v-') \
               and key not in props and key[0].islower():
                props[key] = '__boolean__'

    return props

In [83]:
def extract_component_instances(sfc: str) -> dict[str, list[dict[str, str]]]:
    """Extracts all instances of the PrimeVue component along with their props.

    Returns: {'Button': [{'label': 'OK', ':severity': 'warn'}, ...], ...}
    Multiple instances of the same component → list
    """
    template_match = re.search(r'<template>(.*?)</template>', sfc, re.DOTALL)
    if not template_match:
        return {}

    template = template_match.group(1)
    instances = {}

    for comp in PRIMEVUE_COMPONENTS:
        tag_pattern = re.compile(
            rf'<{comp}((?:\s[^>]*?)?)\s*(?:/>|>)',
            re.DOTALL
        )

        comp_instances = [
            parse_props(m.group(1))
            for m in tag_pattern.finditer(template)
        ]

        if comp_instances:
            instances[comp] = comp_instances

    return instances

In [84]:
def compute_prop_accuracy(
    gen_instances: dict[str, list[dict]],
    gt_instances:  dict[str, list[dict]],
) -> dict:
    """Compares props for correctly identified components (true positives).

    Three categories per prop:
    - correct:      Name and value match the ground truth
    - wrong:        Name matches, but value differs
    - hallucinated: Prop is in the generated code but not in the ground truth
    - missing:      Prop is in the ground truth but not in the generated code
    """
    per_component = {}
    total_correct = total_wrong = total_hallucinated = total_missing = 0

    # Only True Positives (available in both)
    common = set(gen_instances.keys()) & set(gt_instances.keys())

    for comp in common:
        gen_list = gen_instances[comp]
        gt_list  = gt_instances[comp]

        correct = wrong = hallucinated = missing = 0

        for gen_props, gt_props in zip(gen_list, gt_list):
            for key, gt_val in gt_props.items():
                if key in gen_props:
                    if gen_props[key] == gt_val:
                        correct += 1
                    else:
                        wrong += 1
                else:
                    missing += 1

            for key in gen_props:
                if key not in gt_props:
                    hallucinated += 1

        total = correct + wrong + hallucinated + missing
        per_component[comp] = {
            'accuracy':     round(correct / total, 4) if total else 1.0,
            'correct':      correct,
            'wrong':        wrong,
            'hallucinated': hallucinated,
            'missing':      missing,
            'instances':    len(list(zip(gen_list, gt_list))),
        }

        total_correct     += correct
        total_wrong       += wrong
        total_hallucinated+= hallucinated
        total_missing     += missing

    grand_total = total_correct + total_wrong + total_hallucinated + total_missing

    return {
        'overall_accuracy': round(total_correct / grand_total, 4)
                            if grand_total else 1.0,
        'per_component':    per_component,
        'totals': {
            'correct':      total_correct,
            'wrong':        total_wrong,
            'hallucinated': total_hallucinated,
            'missing':      total_missing,
        },
    }

In [85]:
prop_results: list[dict] = []

for complexity in COMPLEXITIES:
    gt_files = GROUPED_FILES.get('gt', {}).get(complexity, {})

    for index, gt_sfc in gt_files.items():
        gt_instances = extract_component_instances(gt_sfc)

        for method in EVAL_METHODS:
            gen_sfc = GROUPED_FILES.get(method, {}).get(complexity, {}).get(index)

            if gen_sfc is None:
                continue

            gen_instances = extract_component_instances(gen_sfc)
            scores        = compute_prop_accuracy(gen_instances, gt_instances)

            prop_results.append({
                'mockup':           f'{complexity}-{index}',
                'complexity':        complexity,
                'index':             index,
                'method':            method,
                'overall_accuracy':  scores['overall_accuracy'],
                'correct':           scores['totals']['correct'],
                'wrong':             scores['totals']['wrong'],
                'hallucinated':      scores['totals']['hallucinated'],
                'missing':           scores['totals']['missing'],
            })

# Aggregation by method
print(f'\n{"Method":6s} {"Ø Accuracy":>11s} {"Ø Correct":>10s} '
      f'{"Ø Halluc.":>10s} {"Ø Missing":>10s}')
print('-' * 52)

by_method_prop = defaultdict(list)
for r in prop_results:
    by_method_prop[r['method']].append(r)

for method in EVAL_METHODS:
    items = by_method_prop[method]

    if not items:
        continue

    n = len(items)

    print(f'{method:6s} '
          f'{sum(i["overall_accuracy"] for i in items)/n:11.3f} '
          f'{sum(i["correct"]      for i in items)/n:10.1f} '
          f'{sum(i["hallucinated"] for i in items)/n:10.1f} '
          f'{sum(i["missing"]      for i in items)/n:10.1f}')

Path('reports/eval_prop_accuracy.json').write_text(
    json.dumps({
        'metric':     'component_prop_accuracy',
        'per_method': {
            method: {
                'avg_accuracy':     round(sum(i['overall_accuracy'] for i in items) / len(items), 4),
                'avg_hallucinated': round(sum(i['hallucinated']     for i in items) / len(items), 2),
                'avg_missing':      round(sum(i['missing']          for i in items) / len(items), 2),
                'n':                len(items),
            }
            for method, items in by_method_prop.items() if items
        },
        'files': prop_results,
    }, indent=4, ensure_ascii=False),
    encoding='utf-8'
)
print('\nSaved: reports/eval_prop_accuracy.json')


Method  Ø Accuracy  Ø Correct  Ø Halluc.  Ø Missing
----------------------------------------------------
a            0.193        3.3        3.6       14.0
b1           0.441        5.7        7.4        8.4
b2           0.260        5.4        8.4       10.3
b3           0.312        6.5        8.6       11.0
c1           0.250        3.7       10.4        9.4
c2           0.240        5.1       11.4       11.1
c3           0.226        5.0       11.4       11.4

Saved: reports/eval_prop_accuracy.json
